In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def aplicar_bag_of_words_por_regras_texto(df, df_regras, corte_porcentagem=30.0):
    df = df.copy()
    df["temas_encontrados"] = [[] for _ in range(len(df))]
    df["termos_capturados"] = [[] for _ in range(len(df))]
    df["acertou_tema"] = False
    
    # Prepara o texto do processo (caixa baixa)
    df["texto_processado"] = df["inteiro_teor_lematizado"].fillna("").astype(str).str.lower()

    # Coluna onde estão as regras com E/OU
    for _, row in df_regras.iterrows():
        regra_bruta = str(row["PALAVRAS-CHAVES"])
        tema_codigo = row["TEMA CÓDIGO"]

        if regra_bruta == "NAN" or pd.isna(regra_bruta) or not regra_bruta.strip():
            continue

        # 1. Divide a regra pelo conectivo " E " para isolar cada parêntese/categoria
        # Ex: ["(Defensoria... OU DP)", "(honorários... OU sucumbência)", ...]
        blocos_categorias = regra_bruta.split(" E ")
        
        categorias_palavras = []
        todas_palavras_do_tema = []
        
        for bloco in blocos_categorias:
            # Remove os parênteses externos do bloco
            bloco_limpo = bloco.replace("(", "").replace(")", "").strip()
            
            # Divide pelo conectivo " OU " para pegar os termos sinônimos
            termos = [termo.strip().lower() for termo in bloco_limpo.split(" OU ") if termo.strip()]
            
            if termos:
                categorias_palavras.append(termos)
                todas_palavras_do_tema.extend(termos)

        total_categorias = len(categorias_palavras)
        if total_categorias == 0:
            continue

        #print(f"Processando Tema {tema_codigo} via BoW ({total_categorias} categorias mapeadas)...")

        # 2. Configura o vetorizador com o vocabulário extraído das regras
        vocabulo_unico = list(set(todas_palavras_do_tema))
        vectorizer = CountVectorizer(
            vocabulary=vocabulo_unico, 
            ngram_range=(1, 5), # Aumentado para 5 para pegar termos longos como "aparelhamento da defensoria pública"
            lowercase=True,
            token_pattern=r'(?u)\b[\w\.\/\-]+\b' # Mantém pontos e barras das leis e súmulas
        )
        
        try:
            X = vectorizer.fit_transform(df["texto_processado"])
            recursos = vectorizer.get_feature_names_out()
        except Exception as e:
            print(f"Erro ao vetorizar tema {tema_codigo}: {e}")
            continue

        matriz_presenca = X.toarray() > 0

        # 3. Varre as linhas do DataFrame procurando os termos
        for idx in range(len(df)):
            palavras_presentes_na_linha = set(recursos[matriz_presenca[idx]])
            
            if not palavras_presentes_na_linha:
                continue

            categorias_atendidas = 0
            termos_encontrados = []

            # Valida quantas das categorias (os antigos "E") têm pelo menos uma palavra presente ("OU")
            for lista_categoria in categorias_palavras:
                intersecao = palavras_presentes_na_linha.intersection(lista_categoria)
                if intersecao:
                    categorias_atendidas += 1
                    termos_encontrados.extend(list(intersecao))

            # 4. Calcula a nota de corte
            porcentagem_sucesso = (categorias_atendidas / total_categorias) * 100

            if porcentagem_sucesso >= corte_porcentagem:
                df.at[df.index[idx], "temas_encontrados"].append(f"{tema_codigo} ({porcentagem_sucesso:.1f}%)")
                df.at[df.index[idx], "termos_capturados"].append(f"{tema_codigo}: {list(set(termos_encontrados))}")
                if df.at[df.index[idx], "TEMA_CODIGO"] == str(tema_codigo):
                    df.at[df.index[idx], "acertou_tema"] = True

    df.drop(columns=["texto_processado"], inplace=True)
    return df

In [2]:
df_lematizado = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\dataset_enriquecido_10062026_lematizado.parquet")
df_temas = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Temas_STF_com_Regex.csv")

In [3]:
df_lematizado_sample = df_lematizado.sample(frac=0.01, random_state=42)
df_sample = df_lematizado_sample.iloc[[0]]

In [8]:
# Chame a função passando os seus DataFrames
df_resultado = aplicar_bag_of_words_por_regras_texto(
    df=df_lematizado_sample, 
    df_regras=df_temas, 
    corte_porcentagem=30.0  # Você pode mudar para 60.0, 80.0, etc.
)

In [ ]:
df_resultado[['TEMA_CODIGO','inteiro_teor_lematizado','temas_encontrados','termos_capturados']].to_csv("")

,TEMA_ORIGEM,TEMA_CODIGO,inteiro_teor,inteiro_teor_lematizado,temas_encontrados,termos_capturados
4013,STJ,1075,Erro Parser>>>>>inicio<<<<<\nAvenida Jamel Cec...,Erro Parser>>>>>inicio < < < < < \n Avenida Ja...,[1002 (30.8%)],"[1002: ['sucumbência', 'fazenda pública', 'ent..."
7839,STJ,960,Erro Parser>>>>>inicio<<<<<\nPROCURAÇÃO AD JUD...,Erro Parser>>>>>inicio < < < < < \n PROCURAÇÃO...,"[1015 (33.3%), 1095 (40.0%)]","[1015: ['recurso repetitivo', 'recurso especia..."
2199,STF,359,>>>>>inicio<<<<<\nRua Dr. Olinto Manso Pereira...,> > > > > inicio < < < < < \n Rua Dr. Olinto M...,[],[]
5981,STJ,1365,>>>>>inicio<<<<<\nEXCELENTÍSSIMO(A) SENHOR(A) ...,> > > > > inicio < < < < < \n EXCELENTÍSSIMO(A...,"[1015 (33.3%), 1069 (33.3%)]","[1015: ['caderneta de poupança'], 1069: ['nega..."
1061,STF,1234,>>>>>inicio<<<<<\nPRESIDÊNCIA DO TRIBUNAL DE J...,> > > > > inicio < < < < < \n pRESIDÊNCIA DO T...,"[1234 (38.9%), 1015 (33.3%), 106 (66.7%)]","[1234: ['ceaf', 'tratamento médico', 'medicame..."
